# Avance 4 — Modelos alternativos individuales

Comparación de ocho estimadores individuales (sin ensambles) para predecir `log_tdpa`: Regresión Lineal, Ridge, Elastic Net, SVR, Árbol de Decisión, K Vecinos, Perceptrón Multicapa (MLP) y una GNN GraphSAGE. Se conserva el preprocesamiento sin fuga y la validación espacial de cinco folds. La métrica principal es RMSE en escala logarítmica; también reportamos MAE, R², WAPE, errores en vehículos por día y tiempos de entrenamiento.

El preprocesamiento se importa desde `tdpa_comun.py`. Mantén ese archivo junto al notebook en `04_NOTEBOOKS`. El corredor de prueba externa queda reservado: no se usa para elegir modelos ni hiperparámetros.


## Paso 1 — Dependencias

Este notebook requiere las dependencias de `requirements.txt` en el entorno del kernel. Instálalas una sola vez antes de ejecutar la libreta completa; el notebook no instala ni actualiza paquetes.


## Paso 2 — Importar librerías y funciones compartidas

In [1]:
from pathlib import Path
import sys
from time import perf_counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics.pairwise import haversine_distances
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from IPython.display import display

# Encontrar la raíz del repositorio sin necesidad de instalar tdpa_comun como paquete.
def buscar_raiz():
    for carpeta in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (carpeta / '03_PROCESSED').is_dir() and (carpeta / '04_NOTEBOOKS').is_dir():
            return carpeta
    raise FileNotFoundError('Abre VS Code en la raíz de tu repositorio (03_PROCESSED y 04_NOTEBOOKS)')

ROOT = buscar_raiz()
sys.path.insert(0, str(ROOT / '04_NOTEBOOKS'))
from tdpa_comun import (SEED, TARGET, PreparadorSeguro, preparar_datos,
                        construir_folds, detectar_log, invertir_log)

THREADS = 2
np.random.seed(SEED)
FE_PATH = ROOT / '03_PROCESSED' / 'FE_TOP_Model_Table.csv'
OUT_DIR = ROOT / '06_RESULTS' / 'AVANCE4' / 'COMPARACION_SIMPLE'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Datos:', FE_PATH)
print('Resultados:', OUT_DIR)


Datos: /Users/caroflores/Documents/ChatGPT/proyecto integrador/Equipo20_EnergyIntelligence/03_PROCESSED/FE_TOP_Model_Table.csv
Resultados: /Users/caroflores/Documents/ChatGPT/proyecto integrador/Equipo20_EnergyIntelligence/06_RESULTS/AVANCE4/COMPARACION_SIMPLE


## Paso 3 — Leer el CSV y reutilizar la validación

`preparar_datos` reserva el corredor de prueba y aplica la purga espacial; `construir_folds` prepara cinco particiones por bloque geográfico. Ambas funciones se importan del módulo común y **no se vuelven a escribir**. El preprocesamiento (`PreparadorSeguro`) seguirá ajustándose dentro de cada fold.


In [2]:
tabla, train_fe, test_fe = preparar_datos(FE_PATH)
transformacion, error_log = detectar_log(train_fe)
folds = construir_folds(train_fe)
print('Transformación objetivo:', transformacion)
display(pd.DataFrame([{'fold': f, 'n_train': len(tr), 'n_valid': len(va)}
                      for f, tr, va in folds]))


Tabla maestra (1075, 274) | train 703 | test externo 23 | purgados 349
Transformación objetivo: log


,fold,n_train,n_valid
0,1,127,141
1,2,149,141
2,3,168,141
3,4,107,140
4,5,120,140


## Paso 4 — Comparación inicial de seis modelos

Todos los algoritmos tabulares usan el mismo `PreparadorSeguro(modo="Filtro + PCA", recortar=False)`, reajustado dentro de cada fold. Cada algoritmo se evalúa con los mismos cinco folds espaciales. El ranking inicial define los dos modelos que pasan al ajuste de hiperparámetros. El corredor externo permanece reservado hasta la evaluación final.


In [3]:
from sklearn.linear_model import ElasticNet, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor


def construir_modelo(nombre, parametros=None):
    p = dict(parametros or {})
    estimadores = {
        "Regresión Lineal": LinearRegression(fit_intercept=p.get("fit_intercept", True)),
        "Ridge": Ridge(alpha=p.get("alpha", 10.0), fit_intercept=p.get("fit_intercept", True), random_state=SEED),
        "Elastic Net": ElasticNet(alpha=p.get("alpha", 0.01), l1_ratio=p.get("l1_ratio", 0.5),
                                  max_iter=10000, random_state=SEED),
        "SVR": SVR(kernel="rbf", C=p.get("C", 10), epsilon=p.get("epsilon", 0.15), gamma=p.get("gamma", "scale")),
        "Árbol de Decisión": DecisionTreeRegressor(max_depth=p.get("max_depth", 10),
                    min_samples_leaf=p.get("min_samples_leaf", 5), random_state=SEED),
        "K Vecinos": KNeighborsRegressor(n_neighbors=p.get("n_neighbors", 15),
                    weights=p.get("weights", "distance"), p=p.get("p", 2), n_jobs=THREADS),
        "MLP": MLPRegressor(hidden_layer_sizes=p.get("hidden_layer_sizes", (64, 32)),
                    alpha=p.get("alpha", 0.001), learning_rate_init=p.get("learning_rate_init", 0.001),
                    max_iter=500, early_stopping=True, n_iter_no_change=25, random_state=SEED),
    }
    return Pipeline([("prep", PreparadorSeguro(modo="Filtro + PCA", recortar=False)),
                     ("modelo", estimadores[nombre])])

MODELOS = ["Regresión Lineal", "Ridge", "Elastic Net", "SVR", "Árbol de Decisión",
           "K Vecinos", "MLP", "GNN (GraphSAGE)"]
print(f"Modelos individuales (sin ensambles): {len(MODELOS)}")
print(", ".join(MODELOS))


Modelos individuales (sin ensambles): 8
Regresión Lineal, Ridge, Elastic Net, SVR, Árbol de Decisión, K Vecinos, MLP, GNN (GraphSAGE)


## Paso 5 — GNN introductoria

GraphSAGE requiere relaciones entre nodos; aquí conectamos los cinco sitios más cercanos usando sus coordenadas. No representa todavía la conectividad real de las carreteras. Para reducir fuga de información, cada fold usa únicamente nodos de entrenamiento como vecinos de las observaciones de validación; las etiquetas de validación no entran al entrenamiento.


In [4]:
try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError("Falta PyTorch: ejecuta %pip install torch en una celda") from e

torch.set_num_threads(1)


def matriz_vecinos(coord_train, coord_nuevos=None, k=5):
    """Fila i = promedio de sus vecinos; nuevos nodos reciben mensajes SOLO de train."""
    geo_tr = np.radians(np.asarray(coord_train, dtype=float))
    dist_tr = haversine_distances(geo_tr, geo_tr)
    np.fill_diagonal(dist_tr, np.inf)
    k = min(k, len(geo_tr)-1)
    if k < 1:
        raise ValueError("La GNN necesita al menos dos sitios de entrenamiento")
    indices_tr = np.argsort(dist_tr, axis=1)[:, :k]
    n = len(geo_tr)
    A = np.zeros((n, n), dtype=np.float32)
    A[np.arange(n)[:, None], indices_tr] = 1.0 / k
    if coord_nuevos is None:
        return torch.tensor(A).to_sparse()

    geo_nuevos = np.radians(np.asarray(coord_nuevos, dtype=float))
    dist_nuevos = haversine_distances(geo_nuevos, geo_tr)
    indices_nuevos = np.argsort(dist_nuevos, axis=1)[:, :k]
    A_todos = np.zeros((n + len(geo_nuevos), n + len(geo_nuevos)), dtype=np.float32)
    A_todos[:n, :n] = A
    A_todos[n + np.arange(len(geo_nuevos))[:, None], indices_nuevos] = 1.0 / k
    return torch.tensor(A_todos).to_sparse()


class GraphSAGE(nn.Module):
    """Dos capas: atributos propios + promedio de atributos de vecinos."""
    def __init__(self, n_features, ocultas=32):
        super().__init__()
        self.capa1 = nn.Linear(2 * n_features, ocultas)
        self.capa2 = nn.Linear(2 * ocultas, 1)
        self.dropout = nn.Dropout(0.15)

    def forward(self, X, A):
        h = torch.relu(self.capa1(torch.cat([X, torch.sparse.mm(A, X)], dim=1)))
        h = self.dropout(h)
        return self.capa2(torch.cat([h, torch.sparse.mm(A, h)], dim=1)).flatten()


def predecir_gnn(X_train, y_train, geo_train, X_valid, geo_valid, epocas=150, ocultas=32, learning_rate=0.01, weight_decay=0.001):
    """Una GNN nueva por fold, sin targets ni aristas entrantes desde validación a train."""
    torch.manual_seed(SEED)
    X_tr = torch.tensor(np.asarray(X_train, dtype=np.float32))
    X_va = torch.tensor(np.asarray(X_valid, dtype=np.float32))
    A_train = matriz_vecinos(geo_train)
    A_todos = matriz_vecinos(geo_train, geo_valid)
    y = np.asarray(y_train, dtype=np.float32)
    media, escala = float(y.mean()), float(max(y.std(), 1e-6))
    y_normalizada = torch.tensor((y - media) / escala)

    red = GraphSAGE(n_features=X_tr.shape[1], ocultas=ocultas)
    optimizador = torch.optim.Adam(red.parameters(), lr=learning_rate, weight_decay=weight_decay)
    for _ in range(epocas):
        red.train()
        optimizador.zero_grad()
        pred = red(X_tr, A_train)
        error = nn.functional.mse_loss(pred, y_normalizada)
        error.backward()
        optimizador.step()

    red.eval()
    with torch.no_grad():
        pred_train = red(X_tr, A_train).numpy() * escala + media
        pred_valid = red(torch.cat([X_tr, X_va]), A_todos).numpy()[-len(X_va):] * escala + media
    return pred_train, pred_valid

## Paso 5 — Entrenar y evaluar en los cinco folds

Para cada modelo entrenamos desde cero dentro de cada fold. El preprocesamiento se ajusta sólo con los datos de entrenamiento de ese fold. Se reportan métricas de entrenamiento y validación; el orden se determina por RMSE logarítmico medio de validación espacial.


In [5]:
def evaluar_prediccion(reales, pred_log, modo_log):
    y = reales[TARGET].to_numpy(float)
    tdpa_real = reales["tdpa"].to_numpy(float)
    tdpa_pred = np.maximum(0, invertir_log(np.asarray(pred_log, dtype=float), modo_log))
    return {
        "rmse_log": np.sqrt(mean_squared_error(y, pred_log)),
        "mae_log": mean_absolute_error(y, pred_log),
        "r2_log": r2_score(y, pred_log),
        "mae_veh_dia": mean_absolute_error(tdpa_real, tdpa_pred),
        "rmse_veh_dia": np.sqrt(mean_squared_error(tdpa_real, tdpa_pred)),
        "wape_pct": 100 * np.abs(tdpa_real - tdpa_pred).sum() / np.abs(tdpa_real).sum(),
    }


def entrenar_predecir(nombre, tr, va, parametros=None):
    parametros = dict(parametros or {})
    if nombre == "GNN (GraphSAGE)":
        prep = PreparadorSeguro(modo="Filtro + PCA", recortar=False)
        X_tr = prep.fit_transform(tr, tr[TARGET])
        X_va = prep.transform(va)
        return predecir_gnn(X_tr, tr[TARGET], tr[["lat", "lon"]], X_va, va[["lat", "lon"]],
                            epocas=parametros.get("epocas", 150), ocultas=parametros.get("ocultas", 32),
                            learning_rate=parametros.get("learning_rate", 0.01),
                            weight_decay=parametros.get("weight_decay", 0.001))
    pipeline = construir_modelo(nombre, parametros)
    pipeline.fit(tr, tr[TARGET])
    return pipeline.predict(tr), pipeline.predict(va)


def evaluar_modelos(datos, folds):
    filas, predicciones = [], []
    for nombre in MODELOS:
        print(f"Entrenando: {nombre}")
        for fold, idx_tr, idx_va in folds:
            tr, va = datos.loc[idx_tr], datos.loc[idx_va]
            inicio = perf_counter()
            p_tr, p_va = entrenar_predecir(nombre, tr, va)
            segundos = perf_counter() - inicio
            for conjunto, reales, estimados in [("train", tr, p_tr), ("valid", va, p_va)]:
                filas.append({"modelo": nombre, "fold": fold, "conjunto": conjunto,
                              "n_sitios": len(reales), "tiempo_seg": segundos,
                              **evaluar_prediccion(reales, estimados, transformacion)})
            predicciones.extend({"modelo": nombre, "fold": fold, "site_id": site,
                                  "real_log": float(real), "pred_log": float(pred)}
                                 for site, real, pred in zip(va.site_id, va[TARGET], p_va))
            print(f"  Fold {fold}: RMSE valid = {filas[-1]['rmse_log']:.4f}")
    return pd.DataFrame(filas), pd.DataFrame(predicciones)

metricas_folds, predicciones_cv = evaluar_modelos(train_fe, folds)
metricas_folds.to_csv(OUT_DIR / "metricas_por_fold.csv", index=False, encoding="utf-8-sig")
predicciones_cv.to_csv(OUT_DIR / "predicciones_cv.csv", index=False, encoding="utf-8-sig")
print("Evaluación terminada. Resultados guardados en:", OUT_DIR)


Entrenando: Regresión Lineal
  Fold 1: RMSE valid = 0.9259


  Fold 2: RMSE valid = 0.9416


  Fold 3: RMSE valid = 1.6981


  Fold 4: RMSE valid = 0.9185
  Fold 5: RMSE valid = 0.7550
Entrenando: Ridge


  Fold 1: RMSE valid = 0.7439
  Fold 2: RMSE valid = 0.7437


...

/opt/anaconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


  Fold 4: RMSE valid = 1.4667
  Fold 5: RMSE valid = 1.7004
Entrenando: GNN (GraphSAGE)


  Fold 1: RMSE valid = 1.0222
  Fold 2: RMSE valid = 0.9059


  Fold 3: RMSE valid = 0.9940
  Fold 4: RMSE valid = 0.8622


  Fold 5: RMSE valid = 0.8786
Evaluación terminada. Resultados guardados en: /Users/caroflores/Documents/ChatGPT/proyecto integrador/Equipo20_EnergyIntelligence/06_RESULTS/AVANCE4/COMPARACION_SIMPLE


## Paso 6 — Comparación inicial y selección de finalistas

La tabla incluye las ocho alternativas individuales y se ordena por RMSE logarítmico promedio de validación espacial. También resume MAE, R², métricas en vehículos por día, WAPE, tiempo de entrenamiento y brecha train-validación. Los dos primeros modelos pasan al ajuste fino. No se emplea ningún ensamble ni se consulta el corredor externo para seleccionar finalistas.


In [6]:
valid = metricas_folds.query("conjunto == 'valid'")
entrenamiento = metricas_folds.query("conjunto == 'train'")
ranking = valid.groupby("modelo").agg(
    rmse_valid=("rmse_log", "mean"),
    sd_rmse=("rmse_log", "std"),
    mae_valid=("mae_log", "mean"),
    r2_valid=("r2_log", "mean"),
    mae_veh_dia=("mae_veh_dia", "mean"),
    rmse_veh_dia=("rmse_veh_dia", "mean"),
    wape_pct=("wape_pct", "mean"),
    tiempo_promedio_s=("tiempo_seg", "mean"),
).join(entrenamiento.groupby("modelo")["rmse_log"].mean().rename("rmse_train"))
ranking["brecha_rmse"] = ranking["rmse_valid"] - ranking["rmse_train"]
ranking = ranking.sort_values("rmse_valid")
ranking.to_csv(OUT_DIR / "ranking_modelos.csv", encoding="utf-8-sig")
display(ranking.round(4))

print("Dos finalistas para ajuste fino:", ", ".join(ranking.index[:2]))


,rmse_valid,sd_rmse,mae_valid,r2_valid,mae_veh_dia,rmse_veh_dia,wape_pct,tiempo_promedio_s,rmse_train,brecha_rmse
modelo,,,,,,,,,,
Ridge,0.7466,0.0219,0.5822,0.5274,1.106698e+04,2.145477e+04,5.683930e+01,0.1786,0.5025,0.2441
Elastic Net,0.7738,0.0471,0.6007,0.4879,1.142910e+04,2.229165e+04,5.872720e+01,0.1310,0.4775,0.2963
SVR,0.8616,0.0794,0.6777,0.3606,1.231939e+04,2.306789e+04,6.326810e+01,0.1329,0.3329,0.5287
GNN (GraphSAGE),0.9326,0.0714,0.7328,0.2589,1.340725e+04,2.470485e+04,6.904590e+01,0.3271,0.1878,0.7448
K Vecinos,0.9719,0.0543,0.7691,0.2014,1.302938e+04,2.440245e+04,6.690910e+01,0.1643,0.0000,0.9719
Árbol de Decisión,0.9953,0.1526,0.7881,0.1327,1.316954e+04,2.474375e+04,6.755180e+01,0.1326,0.4111,0.5842
Regresión Lineal,1.0478,0.3713,0.7145,0.0378,8.755144e+08,1.039601e+10,4.268842e+06,0.2203,0.4423,0.6055
MLP,1.8174,0.5635,1.4001,-1.8614,1.601905e+05,5.718042e+05,7.879228e+02,0.1987,0.9466,0.8708


Dos finalistas para ajuste fino: Ridge, Elastic Net


## Paso 7 — Ajuste fino de los dos mejores modelos

Se prueban hasta 12 configuraciones por finalista, muestreadas de espacios explícitos de hiperparámetros con semilla fija; se incluye la configuración base. Cada configuración se evalúa en los mismos cinco folds espaciales y el preprocesador se vuelve a ajustar dentro de cada fold. El ranking usa RMSE log medio; se guardan también su desviación entre folds, MAE, WAPE y tiempo. El corredor externo sigue reservado.


In [7]:
from itertools import product

# Espacios explícitos; se incluyen los valores base de la comparación inicial.
ESPACIOS = {
    "Regresión Lineal": list(product([True, False],)),
    "Ridge": list(product([0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000, 100000], [True, False])),
    "Elastic Net": list(product([0.0001, 0.001, 0.01, 0.1, 1], [0.1, 0.5, 0.9])),
    "SVR": list(product([1, 10, 50, 100], [0.05, 0.15, 0.3], ["scale", "auto"])),
    "Árbol de Decisión": list(product([3, 5, 10, None], [1, 3, 8])),
    "K Vecinos": list(product([5, 10, 15, 25, 40], ["uniform", "distance"], [1, 2])),
    "MLP": list(product([(32,), (64,), (64, 32), (100, 50)], [0.0001, 0.001, 0.01], [0.0005, 0.001])),
    "GNN (GraphSAGE)": list(product([16, 32, 64], [100, 150, 250], [0.005, 0.01])),
}

def configuraciones(nombre, n=12):
    valores = ESPACIOS[nombre]
    rng = np.random.default_rng(SEED)
    indices = rng.choice(len(valores), size=min(n, len(valores)), replace=False)
    seleccion = [valores[i] for i in indices]
    llaves = {
        "Regresión Lineal": ["fit_intercept"],
        "Ridge": ["alpha", "fit_intercept"],
        "Elastic Net": ["alpha", "l1_ratio"],
        "SVR": ["C", "epsilon", "gamma"],
        "Árbol de Decisión": ["max_depth", "min_samples_leaf"],
        "K Vecinos": ["n_neighbors", "weights", "p"],
        "MLP": ["hidden_layer_sizes", "alpha", "learning_rate_init"],
        "GNN (GraphSAGE)": ["ocultas", "epocas", "learning_rate"],
    }[nombre]
    bases = {
        "Regresión Lineal": (True,), "Ridge": (10, True), "Elastic Net": (0.01, 0.5),
        "SVR": (10, 0.15, "scale"), "Árbol de Decisión": (10, 5),
        "K Vecinos": (15, "distance", 2), "MLP": ((64, 32), 0.001, 0.001),
        "GNN (GraphSAGE)": (32, 150, 0.01),
    }[nombre]
    if bases not in seleccion:
        seleccion[-1] = bases
    return [dict(zip(llaves, c)) for c in seleccion]

def ajustar_finalistas(datos, folds, nombres):
    filas = []
    for nombre in nombres:
        print(f"Ajustando {nombre} ({len(configuraciones(nombre))} configuraciones)")
        for numero, params in enumerate(configuraciones(nombre), 1):
            errores, maes, wapes, tiempos = [], [], [], []
            for fold, idx_tr, idx_va in folds:
                tr, va = datos.loc[idx_tr], datos.loc[idx_va]
                inicio = perf_counter()
                _, p_va = entrenar_predecir(nombre, tr, va, params)
                tiempos.append(perf_counter() - inicio)
                metricas = evaluar_prediccion(va, p_va, transformacion)
                errores.append(metricas["rmse_log"])
                maes.append(metricas["mae_log"])
                wapes.append(metricas["wape_pct"])
            filas.append({"modelo": nombre, "configuracion": numero, "hiperparametros": repr(params),
                          "rmse_valid": float(np.mean(errores)), "sd_rmse": float(np.std(errores, ddof=1)),
                          "mae_valid": float(np.mean(maes)), "wape_pct": float(np.mean(wapes)),
                          "tiempo_promedio_s": float(np.mean(tiempos))})
    return pd.DataFrame(filas).sort_values(["rmse_valid", "mae_valid", "tiempo_promedio_s"])

finalistas = ranking.index[:2].tolist()
resultados_ajuste = ajustar_finalistas(train_fe, folds, finalistas)
resultados_ajuste.to_csv(OUT_DIR / "ajuste_hiperparametros.csv", index=False, encoding="utf-8-sig")
mejores_ajustados = resultados_ajuste.groupby("modelo", sort=False).head(1).sort_values(
    ["rmse_valid", "mae_valid", "tiempo_promedio_s"])
mejores_ajustados.to_csv(OUT_DIR / "mejores_modelos_ajustados.csv", index=False, encoding="utf-8-sig")
display(resultados_ajuste.groupby("modelo", sort=False).head(3).round(4))
display(mejores_ajustados.round(4))

# Regla predefinida: RMSE medio menor; desempates por MAE y tiempo promedio.
modelo_final = mejores_ajustados.iloc[0]
nombre_final = modelo_final["modelo"]
print(f"Modelo individual final para la siguiente etapa: {nombre_final}")
print(f"Configuración: {modelo_final['hiperparametros']}")
print(f"RMSE CV espacial = {modelo_final['rmse_valid']:.4f} ± {modelo_final['sd_rmse']:.4f}; "
      f"MAE log = {modelo_final['mae_valid']:.4f}; WAPE = {modelo_final['wape_pct']:.2f}%; "
      f"tiempo medio = {modelo_final['tiempo_promedio_s']:.2f} s/fold.")
for _, ajustado in mejores_ajustados.iterrows():
    rmse_inicial = float(ranking.loc[ajustado["modelo"], "rmse_valid"])
    cambio_pct = 100 * (rmse_inicial - ajustado["rmse_valid"]) / rmse_inicial
    print(f"Ajuste de {ajustado['modelo']}: RMSE inicial {rmse_inicial:.4f} → "
          f"ajustado {ajustado['rmse_valid']:.4f} ({cambio_pct:+.2f}% de reducción).")
if len(mejores_ajustados) > 1:
    segundo = mejores_ajustados.iloc[1]
    print(f"Justificación: {nombre_final} logra el menor RMSE CV ({modelo_final['rmse_valid']:.4f} frente a "
          f"{segundo['rmse_valid']:.4f} de {segundo['modelo']}), con WAPE {modelo_final['wape_pct']:.2f}% "
          f"y {modelo_final['tiempo_promedio_s']:.2f} s/fold. Ridge es lineal y regularizado, "
          "así que su relación es más sencilla de documentar que la de los modelos no lineales.")
if "Regresión Lineal" in ranking.index and ranking.loc["Regresión Lineal", "mae_veh_dia"] > 1e6:
    print("Nota de comparación: Regresión Lineal extrapola a valores extremos al invertir la transformación; "
          "sus métricas en vehículos/día reflejan esa limitación.")


Ajustando Ridge (12 configuraciones)


Ajustando Elastic Net (12 configuraciones)


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.429e+00, tolerance: 1.176e-02
  model = cd_fast.enet_coordinate_descent(


,modelo,configuracion,hiperparametros,rmse_valid,sd_rmse,mae_valid,wape_pct,tiempo_promedio_s
4,Ridge,5,"{'alpha': 10, 'fit_intercept': True}",0.7466,0.0219,0.5822,56.8393,0.1335
17,Elastic Net,6,"{'alpha': 0.01, 'l1_ratio': 0.9}",0.7704,0.0491,0.5985,58.6075,0.1324
18,Elastic Net,7,"{'alpha': 0.01, 'l1_ratio': 0.5}",0.7738,0.0471,0.6007,58.7272,0.1317
19,Elastic Net,8,"{'alpha': 0.1, 'l1_ratio': 0.5}",0.7787,0.0527,0.6063,56.0898,0.1575
11,Ridge,12,"{'alpha': 1, 'fit_intercept': True}",0.8078,0.0301,0.6280,65.5273,0.1305
0,Ridge,1,"{'alpha': 0.1, 'fit_intercept': True}",0.8494,0.0622,0.6665,69.2461,0.1340


,modelo,configuracion,hiperparametros,rmse_valid,sd_rmse,mae_valid,wape_pct,tiempo_promedio_s
4,Ridge,5,"{'alpha': 10, 'fit_intercept': True}",0.7466,0.0219,0.5822,56.8393,0.1335
17,Elastic Net,6,"{'alpha': 0.01, 'l1_ratio': 0.9}",0.7704,0.0491,0.5985,58.6075,0.1324


Modelo individual final para la siguiente etapa: Ridge
Configuración: {'alpha': 10, 'fit_intercept': True}
RMSE CV espacial = 0.7466 ± 0.0219; MAE log = 0.5822; WAPE = 56.84%; tiempo medio = 0.13 s/fold.
Ajuste de Ridge: RMSE inicial 0.7466 → ajustado 0.7466 (+0.00% de reducción).
Ajuste de Elastic Net: RMSE inicial 0.7738 → ajustado 0.7704 (+0.43% de reducción).
Justificación: Ridge logra el menor RMSE CV (0.7466 frente a 0.7704 de Elastic Net), con WAPE 56.84% y 0.13 s/fold. Ridge es lineal y regularizado, así que su relación es más sencilla de documentar que la de los modelos no lineales.
Nota de comparación: Regresión Lineal extrapola a valores extremos al invertir la transformación; sus métricas en vehículos/día reflejan esa limitación.


## Paso 8 — Gráficas de rendimiento y estabilidad

In [8]:
fig, ejes = plt.subplots(1, 3, figsize=(17, 4.5))
orden = ranking.index.to_list()
pos = np.arange(len(orden))
ejes[0].barh(pos, ranking.loc[orden, "rmse_valid"], xerr=ranking.loc[orden, "sd_rmse"], capsize=3)
ejes[0].axvline(0.7312, color="gray", ls="--", lw=1.2, label="GBoost histórico: 0.7312")
ejes[0].set(yticks=pos, yticklabels=orden, xlabel="RMSE log (CV espacial)", title="Comparación por modelo")
ejes[0].legend(fontsize=8)
ejes[0].invert_yaxis()

ejes[1].bar(pos-0.18, ranking.loc[orden, "rmse_train"], width=0.36, label="Train")
ejes[1].bar(pos+0.18, ranking.loc[orden, "rmse_valid"], width=0.36, label="Validación")
ejes[1].set(xticks=pos, xticklabels=orden, ylabel="RMSE log", title="Posible sobreajuste")
ejes[1].tick_params(axis="x", rotation=65)
ejes[1].legend()

for nombre in orden:
    z = valid.loc[valid.modelo.eq(nombre)].sort_values("fold")
    ejes[2].plot(z.fold, z.rmse_log, marker="o", label=nombre)
ejes[2].set(xlabel="Fold espacial", ylabel="RMSE log", title="Estabilidad entre regiones", xticks=range(1,6))
ejes[2].legend(fontsize=8)
for eje in ejes:
    eje.grid(alpha=0.2, axis="y")
fig.tight_layout()
fig.savefig(OUT_DIR / "comparacion_modelos.png", dpi=160, bbox_inches="tight")
plt.show()

/var/folders/v4/1dylsnkd6cq_jr914fxdwbyw0000gn/T/ipykernel_62173/1721573256.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Paso 9 — Modelo individual final y alcance

Se lleva a la siguiente etapa el modelo individual con menor RMSE log medio después del ajuste; se usan MAE y tiempo medio como criterios de desempate. La elección compara estimadores individuales, por lo que no combina predicciones de varios modelos. Se debe discutir el error, la variación entre regiones y el costo computacional del finalista frente al segundo lugar. El corredor externo se evalúa una sola vez al final y no se usa para cambiar hiperparámetros.


In [9]:
# Evaluación única del modelo ya seleccionado en el corredor reservado.
# No usar estas métricas para cambiar hiperparámetros ni elegir otro modelo.
params_final = eval(modelo_final["hiperparametros"], {"__builtins__": {}}, {})
train_final = train_fe.copy()
validacion_externa = test_fe.copy()
p_tr_final, p_test_final = entrenar_predecir(nombre_final, train_final, validacion_externa, params_final)
metricas_externas = evaluar_prediccion(validacion_externa, p_test_final, transformacion)
metricas_externas["modelo"] = nombre_final
metricas_externas["hiperparametros"] = repr(params_final)
pd.DataFrame([metricas_externas]).to_csv(OUT_DIR / "evaluacion_prueba_externa_final.csv", index=False, encoding="utf-8-sig")
display(pd.DataFrame([metricas_externas]).round(4))
rmse_cv_final = float(modelo_final["rmse_valid"])
print(f"Comparación validación espacial / corredor externo: RMSE log {rmse_cv_final:.4f} / "
      f"{metricas_externas['rmse_log']:.4f}; R² externo {metricas_externas['r2_log']:.4f}; "
      f"n externo = {len(validacion_externa)} sitios.")
print("Interpretación: el error sube y R² externo es negativo, lo que señala transferencia limitada "
      "al corredor reservado. Se reporta como limitación; no se cambian modelo ni hiperparámetros "
      "con esta prueba externa.")


,rmse_log,mae_log,r2_log,mae_veh_dia,rmse_veh_dia,wape_pct,modelo,hiperparametros
0,0.8854,0.7267,-0.2736,19796.9029,27148.495,55.0028,Ridge,"{'alpha': 10, 'fit_intercept': True}"


Comparación validación espacial / corredor externo: RMSE log 0.7466 / 0.8854; R² externo -0.2736; n externo = 23 sitios.
Interpretación: el error sube y R² externo es negativo, lo que señala transferencia limitada al corredor reservado. Se reporta como limitación; no se cambian modelo ni hiperparámetros con esta prueba externa.


### Interpretación de la prueba externa

La evaluación del corredor reservado da un RMSE mayor que la validación espacial y un R² negativo. Esto indica que el modelo elegido en validación tiene una capacidad de transferencia limitada a este corredor. Se conserva la selección basada en validación y se informa esta diferencia como una limitación; la prueba externa no se usa para reajustar ni cambiar el modelo.


## Declaración sobre el uso de herramientas de inteligencia artificial

Se utilizó ChatGPT como apoyo para revisar los requisitos de la actividad, organizar y editar el código del notebook, proponer configuraciones de hiperparámetros y mejorar la claridad de las explicaciones. El equipo es responsable de revisar, ejecutar, interpretar y validar el código, los resultados y las conclusiones antes de la entrega.
